In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GRU, Dense, Input, Dropout, BatchNormalization, LeakyReLU, ReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report
import os

# הגדרת פרמטרים
WINDOW_SIZE = 30
RANDOM_STATE = 42
BATCH_SIZE = 32
EPOCHS = 300
LEARNING_RATE = 0.001

def create_dominance_labels(df, knee_threshold=0.1, elbow_threshold=0.1):
    """
    יצירת תוויות דומיננטיות עם 3 מחלקות:
    0 = Right, 1 = Left, 2 = No Dominance
    """
    df['dominant_knee'] = np.where(
        abs(df['right_knee_score'] - df['left_knee_score']) < knee_threshold,
        2,
        np.where(df['right_knee_score'] > df['left_knee_score'], 0, 1)
    )
    df['dominant_elbow'] = np.where(
        abs(df['right_elbow_score'] - df['left_elbow_score']) < elbow_threshold,
        2,
        np.where(df['right_elbow_score'] > df['left_elbow_score'], 0, 1)
    )
    return df

def load_and_preprocess_data(file_paths, window_size=WINDOW_SIZE):
    """
    טעינה ועיבוד מקדים של הנתונים עם יצירת תוויות דומיננטיות
    """
    all_data = []
    test_data = []

    for file_path in file_paths:
        df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
        
        required_columns = ['video_id', 'frame', 'window_id', 
                            'overall_movement_score', 
                            'right_knee_score', 'left_knee_score',
                            'right_elbow_score', 'left_elbow_score']
        if not all(col in df.columns for col in required_columns):
            print(f"The file {file_path} is missing required columns. Skipping...")
            continue
        
        df = create_dominance_labels(df)
        
        test_subset = df[df['video_id'].isin([43, 44])]
        if not test_subset.empty:
            test_data.append(test_subset)
        
        df = df[~df['video_id'].isin([43, 44])]
        all_data.append(df)
    
    if not all_data:
        raise ValueError("No valid files found for processing")
    
    data = pd.concat(all_data, ignore_index=True)
    return data, test_data

def prepare_sequences(data, window_size=WINDOW_SIZE):
    """
    הכנת רצפי נתונים עבור GRU לפי חלונות
    """
    feature_columns = [col for col in data.columns if any(x in col for x in ['_x', '_y', '_confidence'])]

    X, y_movement, y_knee, y_elbow = [], [], [], []

    groups = data.groupby(['video_id', 'window_id'])
    
    for (_, _), group in groups:
        if len(group) < window_size:
            continue
        for i in range(0, len(group) - window_size + 1):
            sequence = group.iloc[i:i+window_size][feature_columns].values
            target_frame = group.iloc[i+window_size-1]
            X.append(sequence)
            y_movement.append(target_frame['overall_movement_score'])
            y_knee.append(target_frame['dominant_knee'])
            y_elbow.append(target_frame['dominant_elbow'])

    return (np.array(X), np.array(y_movement), np.array(y_knee), np.array(y_elbow), feature_columns)

def build_gru_model(input_shape, num_knee_classes, num_elbow_classes):
    """
    בניית מודל GRU עם פלט מרובה
    """
    inputs = Input(shape=input_shape)
    
    x = GRU(128, return_sequences=True)(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = GRU(64)(x)
    x = BatchNormalization()(x)
    shared_features = Dropout(0.3)(x)
    
    # Movement output (regression)
    movement_output = Dense(32, activation='relu')(shared_features)
    movement_output = Dense(1)(movement_output)
    movement_output = ReLU(max_value=10, name='movement_score')(movement_output)
    
    # Dominant knee (classification)
    knee_output = Dense(32)(shared_features)
    knee_output = LeakyReLU(alpha=0.01)(knee_output)
    knee_output = Dense(num_knee_classes, activation='softmax', name='dominant_knee')(knee_output)
    
    # Dominant elbow (classification)
    elbow_output = Dense(32)(shared_features)
    elbow_output = LeakyReLU(alpha=0.01)(elbow_output)
    elbow_output = Dense(num_elbow_classes, activation='softmax', name='dominant_elbow')(elbow_output)
    
    model = Model(inputs=inputs, outputs=[movement_output, knee_output, elbow_output])
    
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss={
            'movement_score': 'mean_squared_error',
            'dominant_knee': 'sparse_categorical_crossentropy',
            'dominant_elbow': 'sparse_categorical_crossentropy'
        },
        metrics={
            'movement_score': ['mae'],
            'dominant_knee': ['accuracy'],
            'dominant_elbow': ['accuracy']
        }
    )
    return model

def train_model(X_train, y_train, X_val, y_val, input_shape, num_knee_classes, num_elbow_classes):
    """
    אימון המודל עם callbacks
    """
    model = build_gru_model(input_shape, num_knee_classes, num_elbow_classes)
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)
    model_checkpoint = ModelCheckpoint(
        'best_movenet_infant_movement_model_3class.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
    
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping, reduce_lr, model_checkpoint],
        verbose=1
    )
    return model, history

def evaluate_model(model, X_test, y_test_movement, y_test_knee, y_test_elbow):
    """
    הערכת המודל והצגת מדדים
    """
    y_pred_movement, y_pred_knee_prob, y_pred_elbow_prob = model.predict(X_test)
    y_pred_knee = np.argmax(y_pred_knee_prob, axis=1)
    y_pred_elbow = np.argmax(y_pred_elbow_prob, axis=1)
    
    rmse_movement = np.sqrt(mean_squared_error(y_test_movement, y_pred_movement))
    knee_accuracy = accuracy_score(y_test_knee, y_pred_knee)
    elbow_accuracy = accuracy_score(y_test_elbow, y_pred_elbow)
    
    print("\n===== תוצאות המודל =====")
    print(f"RMSE for movement score: {rmse_movement:.4f}")
    print(f"Knee Accuracy: {knee_accuracy:.4f}")
    print(f"Elbow Accuracy: {elbow_accuracy:.4f}")
    
    print("\nClassification Report - Knee:")
    print(classification_report(
        y_test_knee, y_pred_knee,
        labels=[0, 1, 2],
        target_names=["Right", "Left", "No Dominance"],
        zero_division=0
    ))
    
    print("\nClassification Report - Elbow:")
    print(classification_report(
        y_test_elbow, y_pred_elbow,
        labels=[0, 1, 2],
        target_names=["Right", "Left", "No Dominance"],
        zero_division=0
    ))
    
    return {
        'rmse_movement': rmse_movement,
        'knee_accuracy': knee_accuracy,
        'elbow_accuracy': elbow_accuracy
    }

def main():
    file_paths = ['movenet_dataset.csv']
    
    if not os.path.exists(file_paths[0]):
        print(f"שגיאה: הקובץ '{file_paths[0]}' לא נמצא")
        return
    
    data, test_data = load_and_preprocess_data(file_paths)
    print(f"Loaded {len(data)} rows of data.")
    
    if test_data:
        test_df = pd.concat(test_data, ignore_index=True)
        test_df.to_csv('test_movenet_dataset_3class.csv', index=False)
        print("Saved test dataset as 'test_movenet_dataset_3class.csv'")
    
    X, y_movement, y_knee, y_elbow, feature_columns = prepare_sequences(data)
    
    print(f"Number of sequences: {len(X)}, Features: {X.shape[2]}")
    
    # ✅ תמיד מאמנים עם 3 מחלקות
    num_knee_classes = 3
    num_elbow_classes = 3
    
    X_train_val, X_test, y_movement_train_val, y_movement_test, y_knee_train_val, y_knee_test, y_elbow_train_val, y_elbow_test = train_test_split(
        X, y_movement, y_knee, y_elbow, test_size=0.2, random_state=RANDOM_STATE
    )
    
    X_train, X_val, y_movement_train, y_movement_val, y_knee_train, y_knee_val, y_elbow_train, y_elbow_val = train_test_split(
        X_train_val, y_movement_train_val, y_knee_train_val, y_elbow_train_val, test_size=0.2, random_state=RANDOM_STATE
    )
    
    input_shape = (X_train.shape[1], X_train.shape[2])
    
    y_train = [y_movement_train, y_knee_train, y_elbow_train]
    y_val = [y_movement_val, y_knee_val, y_elbow_val]
    
    model, history = train_model(X_train, y_train, X_val, y_val, input_shape, num_knee_classes, num_elbow_classes)
    
    results = evaluate_model(model, X_test, y_movement_test, y_knee_test, y_elbow_test)
    
    model.save('infant_movenet_movement_gru_model_3class.keras')
    print("Saved final model as 'infant_movenet_movement_gru_model_3class.keras'")
    
    with open('model_features_3class.txt', 'w') as f:
        f.write('\n'.join(feature_columns))
    
    print("Training finished successfully.")

if __name__ == "__main__":
    main()



Loaded 55117 rows of data.
Saved test dataset as 'test_movenet_dataset_3class.csv'
Number of sequences: 28174, Features: 40
Epoch 1/300


C:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


564/564 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - dominant_elbow_accuracy: 0.8874 - dominant_elbow_loss: 0.4067 - dominant_knee_accuracy: 0.8048 - dominant_knee_loss: 0.5656 - loss: 24.9992 - movement_score_loss: 24.0269 - movement_score_mae: 3.9228
Epoch 1: val_loss improved from inf to 8.23990, saving model to best_movenet_infant_movement_model_3class.keras
564/564 ━━━━━━━━━━━━━━━━━━━━ 72s 116ms/step - dominant_elbow_accuracy: 0.8874 - dominant_elbow_loss: 0.4065 - dominant_knee_accuracy: 0.8049 - dominant_knee_loss: 0.5654 - loss: 24.9812 - movement_score_loss: 24.0092 - movement_score_mae: 3.9212 - val_dominant_elbow_accuracy: 0.9148 - val_dominant_elbow_loss: 0.2947 - val_dominant_knee_accuracy: 0.8991 - val_dominant_knee_loss: 0.3298 - val_loss: 8.2399 - val_movement_score_loss: 7.6152 - val_movement_score_mae: 2.4249 - learning_rate: 0.0010
Epoch 2/300
564/564 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - dominant_elbow_accuracy: 0.9219 - dominant_elbow_loss: 0.2865 - dominant_knee_accuracy: 

OSError: [Errno 28] No space left on device